# Representing Text as Data

<a target="_blank" href="https://colab.research.google.com/github/Magallanes-at-UTDT/texts/blob/main/Representing_Text_as_Data.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Social scientists often work with speeches, interviews, manifestos, tweets, news articles, and policy documents.

These materials contain words. Most analytical methods, however, require numbers.

So before we can analyze text, we must answer a basic question:

> **How should we represent text as data?**

This notebook shows that there is no single automatic answer.

We will use the same small corpus and transform it in several different ways.


## Learning goals

By the end of this short notebook, you should be able to:

- explain why text must be converted into numbers;
- recognize that the same documents can produce different matrices;
- distinguish between a raw DTM, a cleaned DTM, an LLM-assisted DTM, and an embedding matrix;
- understand that text representation is a research decision.


## One corpus, several representations

We will use six short political statements.

The documents will not change.

Only the way we represent them will change.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


data = {
    "doc_id": [1, 2, 3, 4, 5, 6],
    "party": [
        "Conservative",
        "Green",
        "Progressive",
        "Nationalist",
        "Liberal",
        "Fiscal Conservative"
    ],
    "text": [
        "We will lower taxes and create more jobs for the working class.",
        "The government must invest heavily in renewable energy and infrastructure.",
        "Healthcare should be a universal right, not a privilege for the wealthy.",
        "We promise to strengthen our borders and enforce existing immigration laws.",
        "Education funding will be increased to ensure every child has equal opportunities.",
        "Our party will cut government spending and reduce the national debt."
    ]
}

manifestos = pd.DataFrame(data)
manifestos


For most of the models:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Raw DTM with CountVectorizer

A **Document-Term Matrix**, or **DTM**, is a table where:

- each row is a document;
- each column is a word;
- each cell shows how many times that word appears in the document.

In this first version, every different word becomes a different column.


In [ ]:


# still performs minimal preprocessing by lowercasing the text, 
# tokenizing words, removing punctuation, and ignoring single-character tokens. 
# No linguistic preprocessing such as stop-word removal or lemmatization is applied.
raw_vectorizer = CountVectorizer()



dtm_raw = raw_vectorizer.fit_transform(manifestos["text"])

dtm_raw_df = pd.DataFrame(
    dtm_raw.toarray(),
    index=manifestos["party"],
    columns=raw_vectorizer.get_feature_names_out()
)

print("Raw DTM shape:", dtm_raw_df.shape)
dtm_raw_df


## What did the computer do?

The computer separated the text into words, changed words to lowercase, ignored punctuation, and counted each word.

But it did not understand meaning.

Related forms such as `tax`, `taxes`, and `taxation` would appear as different columns.


# 2. Classical NLP DTM with spaCy

A second researcher may decide to clean and simplify the language before building the matrix.

For example:

- `taxes` may become `tax`;
- `running` may become `run`;
- common words such as `the`, `and`, and `of` may be removed.

This process is called **preprocessing**.

Here we use **spaCy**, a classical Natural Language Processing library.


In [ ]:
# Run this once in a new Colab session.
# !pip -q install spacy
# !python -m spacy download en_core_web_sm -q
# !python -m spacy download es_core_news_sm -q # spanish

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
# nlp = spacy.load("es_core_news_sm",disable=["ner", "parser"]) # spanish

spacy_texts = []

for text in manifestos["text"]:
    doc = nlp(text)

    lemmas = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and len(token.lemma_) > 2
    ]

    spacy_texts.append(" ".join(lemmas))

spacy_comparison = manifestos[["party", "text"]].copy()
spacy_comparison["spacy_text"] = spacy_texts
spacy_comparison


In [ ]:
spacy_vectorizer = CountVectorizer()
dtm_spacy = spacy_vectorizer.fit_transform(spacy_texts)

dtm_spacy_df = pd.DataFrame(
    dtm_spacy.toarray(),
    index=manifestos["party"],
    columns=spacy_vectorizer.get_feature_names_out()
)

print("spaCy DTM shape:", dtm_spacy_df.shape)
dtm_spacy_df


## Same documents, different matrix

The corpus is exactly the same.

But the vocabulary changed because the preprocessing rules changed.

This means that preprocessing is not only a technical step. It is also a research decision.


# 3. LLM-assisted DTM

A third researcher may ask a Large Language Model, or **LLM**, to prepare the text.

The task is still simple:

> Take each document, lemmatize it, and then build a DTM.

We will show two routes:

- **Ollama**, which runs locally;
- **Gemini**, which uses a remote API.


## 3A. Ollama: local LLM

Ollama runs models on your own computer.

Advantages:

- no API key;
- no daily request limit;
- no token quota;
- the text stays on the local computer.

Requirement:

- Ollama must be installed and running;
- the model must already be downloaded. Download it from [here](https://ollama.com/download?utm_source=chatgpt.com). Run it after installation.


In [ ]:
# input

SYSTEM_RULES = """
Convert the text into keywords for a Document-Term Matrix.

Output should contain only normalized nouns and verbs. 

Turn any noun to its singular form, and verbs to infinitive.

Remove stop words.

Return a single line of lowercase words separated by spaces.

Do not explain your answer.
"""

PROMPT = """
Normalize this text for a Document-Term Matrix:

{text}
"""

In [ ]:
# Install only the Python client.
# Ollama itself must be installed separately and running.
!pip -q install ollama
!ollama pull llama3.2

In [ ]:
import ollama

MODEL = "llama3.2"



ollama_texts = []

for text in manifestos["text"]:

    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_RULES},
            {"role": "user", "content": PROMPT.format(text=text)}
        ],
        options={"temperature": 0.0}
    )

    clean_text = response["message"]["content"].strip()
    ollama_texts.append(clean_text)

ollama_comparison = manifestos[["party", "text"]].copy()
ollama_comparison["ollama_text"] = ollama_texts

ollama_comparison

In [ ]:
ollama_vectorizer = CountVectorizer()
dtm_ollama = ollama_vectorizer.fit_transform(ollama_texts)

dtm_ollama_df = pd.DataFrame(
    dtm_ollama.toarray(),
    index=manifestos["party"],
    columns=ollama_vectorizer.get_feature_names_out()
)

print("Ollama DTM shape:", dtm_ollama_df.shape)
dtm_ollama_df


In [ ]:
# dtm_ollama_df.columns.tolist()

## 3B. Gemini: remote LLM

Gemini performs the same kind of task, but through a cloud service.

Advantages:

- no local model is needed;
- it can be used from Colab.

Limitations:

- an API key is required;
- internet access is required;
- there may be daily limits;
- there may be token limits;
- available model names may change.


In [ ]:
# Run once in a new Colab session.
# !pip -q install google-genai


In [ ]:
from google import genai


GEMINI_MODEL = "gemini-3.6-flash"
API_KEY = Path("sometext.txt").read_text().strip()
client = genai.Client(api_key=API_KEY)


gemini_texts = []

for text in manifestos["text"]:
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=PROMPT.format(text=text),
        config={
            "system_instruction": SYSTEM_RULES,
            "temperature": 0
        }
    )

    clean_text = response.text.strip()
    gemini_texts.append(clean_text)

gemini_comparison = manifestos[["party", "text"]].copy()
gemini_comparison["gemini_text"] = gemini_texts
gemini_comparison


In [ ]:
gemini_vectorizer = CountVectorizer()
dtm_gemini = gemini_vectorizer.fit_transform(gemini_texts)

dtm_gemini_df = pd.DataFrame(
    dtm_gemini.toarray(),
    index=manifestos["party"],
    columns=gemini_vectorizer.get_feature_names_out()
)

print("Gemini DTM shape:", dtm_gemini_df.shape)
dtm_gemini_df


## What should we notice?

Ollama and Gemini are both LLMs, but they may not return exactly the same text.

Their answers depend on the model, prompt, system instruction, model version, and software settings.

Using an LLM does not remove research decisions. It moves some of those decisions into the prompt and the model.


# 4. Embedding Matrix with Sentence Transformers

Now comes a very different idea.

What if we stop using words as columns?

An **embedding** represents each complete document with a list of numbers.

These numbers try to capture the general meaning of the document.

The columns are not words. They are hidden numerical features learned by a language model.


In [ ]:
# Run once in a new Colab session.
!pip -q install sentence-transformers


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
# embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

embedding_matrix = embedding_model.encode(
    manifestos["text"].tolist()
)

print("Embedding matrix shape:", embedding_matrix.shape)


The result contains one row for each document and 384 numerical columns.

But the columns are not words such as `tax`, `education`, or `government`.

They are numerical dimensions.


In [ ]:
embedding_preview = pd.DataFrame(
    embedding_matrix[:, :8],
    index=manifestos["party"],
    columns=[f"dimension_{i}" for i in range(1, 9)]
)

embedding_preview.round(3)


## Where are the words?

There are no word columns.

The matrix contains numbers such as `0.231`, `-0.184`, and `0.507`.

We usually cannot explain each number by itself.

But all the numbers together represent information about the meaning of the document.


# Comparing the representations

We started with the same six documents.

We produced several different numerical datasets.


In [ ]:
representation_summary = pd.DataFrame({
    "representation": [
        "Raw DTM",
        "spaCy DTM",
        "Ollama DTM",
        # "Gemini DTM",
        "Embedding matrix"
    ],
    "rows": [
        dtm_raw_df.shape[0],
        dtm_spacy_df.shape[0],
        dtm_ollama_df.shape[0],
        # dtm_gemini_df.shape[0],
        embedding_matrix.shape[0]
    ],
    "columns": [
        dtm_raw_df.shape[1],
        dtm_spacy_df.shape[1],
        dtm_ollama_df.shape[1],
        # dtm_gemini_df.shape[1],
        embedding_matrix.shape[1]
    ],
    "columns_represent": [
        "Words",
        "Cleaned lemmas",
        "LLM-normalized terms",
        # "LLM-normalized terms",
        "Numerical features"
    ]
})

representation_summary


## Simple comparison

| Representation | What are the columns? | Easy to interpret? | Main idea |
|---|---|---:|---|
| Raw DTM | Words | Yes | Count the words as they appear |
| spaCy DTM | Cleaned lemmas | Yes | Apply linguistic rules first |
| Ollama DTM | LLM-prepared terms | Usually | Ask a local LLM to prepare the text |
| Gemini DTM | LLM-prepared terms | Usually | Ask a remote LLM to prepare the text |
| Embedding matrix | Numerical features | No | Represent the overall meaning |

No method is automatically the best method.

Each representation keeps some information and loses other information.


# The main lesson

Imagine asking four research assistants to prepare the same collection of documents.

One counts every word.

Another cleans the language first.

A third asks an AI to standardize the text.

A fourth ignores individual words and represents each document through its overall meaning.

They all begin with the same documents.

But they produce different datasets.

> **Text analysis does not begin when we run an algorithm. It begins when we decide how text will become data.**


## What comes next?

Now that text has become a matrix, we can begin the analytics.

For example:

- compare documents;
- identify similar documents;
- find important words;
- create groups;
- identify topics;
- build prediction models.

But every later result depends partly on the representation chosen here.


In [ ]:
PROCESSED_DIR = Path("data/processed")
 

dtm_raw_df.to_parquet(PROCESSED_DIR / "dtm_raw.parquet")
dtm_spacy_df.to_parquet(PROCESSED_DIR / "dtm_spacy.parquet")
dtm_ollama_df.to_parquet(PROCESSED_DIR / "dtm_ollama.parquet")
# dtm_gemini_df.to_parquet(PROCESSED_DIR / "dtm_gemini.parquet")

####
embeddings_df = pd.DataFrame(
    embedding_matrix,
    index=manifestos["party"],
    columns=[f"dimension_{i}" for i in range(embedding_matrix.shape[1])]
)

embeddings_df.to_parquet(PROCESSED_DIR / "embeddings.parquet")